In [ ]:
import pickle
from canopy.core import CDTNode
from canopy.llm import ClaudeCodeAdapter, generate, set_adapter
from canopy.wikify import wikify_profile, wikify_tree
from tqdm.auto import tqdm

In [ ]:
set_adapter(ClaudeCodeAdapter(default_model="claude-haiku-4-5"))

In [ ]:
class _LegacyUnpickler(pickle.Unpickler):
    def find_class(self, module: str, name: str) -> type:
        if name == "CDT_Node" and module in ("__main__", "codified_decision_tree", "canopy.core"):
            return CDTNode
        return super().find_class(module, name)


def load_cdt_package(path: str) -> dict:
    with open(path, "rb") as f:
        return _LegacyUnpickler(f).load()

In [ ]:
character = "戸山香澄"
cdt_id = "Kasumi"
lang = "Chinese"
note = '''
Kasumi -> 戸山香澄
Arisa -> 市谷有咲
Rimi -> 牛込里美
Tae -> 花园多惠
Saaya -> 山吹沙绫
'''

cdts = load_cdt_package(f"packages/{cdt_id}.cdt.v3.1.package.relation.pkl")
topic2cdt = cdts["topic2cdt"]
rel_topic2cdt = cdts["rel_topic2cdt"]

In [ ]:
markdown = wikify_profile(topic2cdt, rel_topic2cdt, character=cdt_id)
print(markdown)

In [ ]:
def wikify_llm(character, content, cdt_tree_verbalized, attribute, note, lang="English"):
    """Generate a wiki-style section from CDT tree via LLM."""
    prompt = f'''# Wiki:
{content}

# Pseudo code

{cdt_tree_verbalized}

# Task
The pseudo code above describes the character behavior logic of {character}. Summarize the information from the given code for "{attribute}" section in a wiki style to entail the given character Wiki.
{note}
Output in the following format (using {lang} and the name: {character}):
- {attribute} (The title should also be translated to target language: {lang}) -

{{Content}}'''
    return generate(prompt)

In [ ]:
content = character
for topic in tqdm(topic2cdt, desc="Wikifying attribute information..."):
    cdt = topic2cdt[topic]
    increment = wikify_llm(character, content, cdt.verbalize(), topic, note, lang)
    content = "\n\n".join([content, increment])

for rel_topic in tqdm(rel_topic2cdt, desc="Wikifying interactive information..."):
    cdt = rel_topic2cdt[rel_topic]
    increment = wikify_llm(character, content, cdt.verbalize(), rel_topic, note, lang)
    content = "\n\n".join([content, increment])

In [ ]:
print(content)

In [ ]:
with open(f"profiles/{character}.wikified.profile.txt", "w", encoding="utf-8") as f:
    f.write(content)
print(f"Saved to profiles/{character}.wikified.profile.txt")